# 03. Big Data Analysis: Answering Core Business Questions
**Objective:** Answer all 13 required business and market questions using genuine PySpark aggregations.

## Setup & Spark Initialization

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.spark_session import get_spark_session
from src.data_loader import load_raw_data_spark
from src.data_cleaning import clean_dataset_spark
from src.spark_analysis import *
from pyspark.sql import functions as F

spark = get_spark_session('Zomato_Analysis_Notebook')
raw_df = load_raw_data_spark(spark)
df, _ = clean_dataset_spark(raw_df, deduplicate=True)
df.cache()
print(f'Ready! Cleaned unique restaurants available: {df.count():,}')

### Q1: Total Number of Valid Restaurant Records
How many total valid dining establishments exist after deduplicating multi-zone listings?

In [ ]:
total_restaurants = df.count()
valid_ratings = df.filter(F.col('rating_num').isNotNull()).count()
print(f'Total Valid Restaurants: {total_restaurants:,}')
print(f'Restaurants with Customer Ratings: {valid_ratings:,} ({valid_ratings/total_restaurants*100:.1f}%)')

### Q2 & Q3: Restaurant Distribution by Locality & Top 10 Localities
Which Bangalore commercial corridors have the greatest concentration of restaurants?

In [ ]:
locality_df = analyze_localities(df)
locality_df.show(10, truncate=False)

### Q4: Most Common Cuisines in Bangalore
What are the most widely available culinary categories across all restaurants?

In [ ]:
cuisines_df = analyze_cuisines_popularity(df)
cuisines_df.show(10, truncate=False)

### Q5: Cuisine-Wise Average Ratings with Sample-Size Threshold
We impose a strict minimum threshold (>= 30 restaurants) to eliminate misleading small-sample outliers.

In [ ]:
ranked_cuisines = analyze_cuisine_ratings_with_threshold(df, min_restaurants=30)
ranked_cuisines.show(10, truncate=False)

### Q6 & Q7: Rating and Cost Distributions
Analyzing how restaurant ratings and average meal prices are distributed.

In [ ]:
print('Rating Distribution:')
analyze_rating_distribution(df).show()
print('Cost Distribution by Price Tier:')
analyze_cost_distribution(df).show()

### Q8 & Q9: Approximate Cost vs. Ratings & Votes Correlation
Evaluating statistical correlation between pricing, engagement, and customer satisfaction.

In [ ]:
correlations = analyze_cost_vs_rating_correlation(df)
print('Pearson Correlation Matrix:')
for k, v in correlations.items():
    print(f'  {k}: {v}')

### Q10: Impact of Online Ordering Availability
Comparing performance metrics between restaurants offering online ordering vs. offline-only.

In [ ]:
analyze_online_order_impact(df).show()

### Q11: Impact of Table Booking Reservations
Evaluating customer rating and pricing differences for table-booking enabled venues.

In [ ]:
analyze_table_booking_impact(df).show()

### Q12: High-Engagement Restaurants
Identifying premier Bangalore dining destinations ranked by engagement score: `rating * log10(votes + 1)`.

In [ ]:
high_eng = analyze_high_engagement_restaurants(df, top_n=10)
high_eng.select('name', 'location', 'rating_num', 'votes_num', 'engagement_score').show(10, truncate=False)

### Q13: Data Quality & Completeness Summary
Summary of cleaning decisions and final data completeness percentages.

In [ ]:
import pandas as pd
from src.config import ANALYSIS_RESULTS_DIR
pd.read_csv(ANALYSIS_RESULTS_DIR / 'data_quality_report.csv')